In [1]:
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

# set directories
WD_junxi = Path('PATH_TO_DATA')
WD = WD_junxi
data_dir = Path(WD/'EntTemplates/Analysis/python_Patent/data/')
output_dir = Path(WD/'EntTemplates/Analysis/python_Patent/output/')
# Ensure prediction output directory exists
pred_output_dir = output_dir / "patent_predictions_new"
pred_output_dir.mkdir(parents=True, exist_ok=True)
patent_model_dir = Path("PATH_TO_PATENT_MODEL_FOLDERS")

BERT_output_dir = Path(WD / 'EntTemplates/Analysis/python_BERT/data_patent/positive/')
# disable WANDB

import os
os.environ["WANDB_DISABLED"] = "true" 


In [2]:
df = pd.read_csv(BERT_output_dir / 'AgTechAgbiotechAnimalbiotech_patent_positive_bert.csv', dtype=str)
df_pred = df[["patent_id", "patent_abstract"]].copy()
df_pred = df_pred.rename(columns={"patent_id": "appid", "patent_abstract": "abstract"})

df_pred["appid"] = df_pred["appid"].astype(str)
df_pred["abstract"] = df_pred["abstract"].fillna("").astype(str)
df_pred = df_pred[df_pred["abstract"].str.strip() != ""].reset_index(drop=True)

In [3]:


# Discover all subsegment model folders
model_base = patent_model_dir
model_folders = [p for p in model_base.glob("SUBSEGMENT-*") if (p / "best_model" / "model.safetensors").exists()]
# sort by subsegment name for consistent processing order
model_folders = sorted(model_folders, key=lambda p: p.name)
print(f"Discovered {len(model_folders)} subsegment model folders.")
results_collection = []

# Use the original base tokenizer (not saved in best_model)
tokenizer_base = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_base)

hf_ds = Dataset.from_pandas(df_pred, preserve_index=False)

def safe_tokenize_function(examples):
    return tokenizer(examples["abstract"], padding="max_length", truncation=True, max_length=512)

predict_dataset = hf_ds.map(safe_tokenize_function, batched=True, remove_columns=hf_ds.column_names)

for model_root in model_folders:
    subsegment = model_root.name.replace("SUBSEGMENT-", "")
    out_path_to_check = BERT_output_dir / f"{subsegment}_patent_positive_bert.csv"

    # Skip if results already exist
    if out_path_to_check.exists():
        print(f"Skipping {subsegment}: output already exists at {out_path_to_check.name}")
        continue

    print(f"Processing subsegment: {subsegment}. Total number of sectors: {len(model_folders)}")
    model_path = model_root / "best_model"  # weights live here

    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    trainer = Trainer(model=model)

    predictions = trainer.predict(predict_dataset)
    logits = predictions.predictions
    y_neg = logits[:, 0]
    y_pos = logits[:, 1]
    y_pred = np.argmax(logits, axis=-1)

    df_results = pd.DataFrame({
        "appid": df_pred["appid"],
        "abstract": df_pred["abstract"],
        "pred_pos": y_pos,
        "pred_neg": y_neg,
        "positive": y_pred,
        "sector": subsegment,
    })
    out_path = pred_output_dir / f"{subsegment}_patent_positive_bert.csv"
    df_results.to_csv(out_path, index=False)
    results_collection.append((subsegment, df_results.head()))

# results_collection now holds (subsegment, preview_df) tuples for quick inspection
results_collection

Discovered 217 subsegment model folders.


Map:   0%|          | 0/99988 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Skipping AIMLAIMLSemiconductorsEdgeAISoftware: output already exists at AIMLAIMLSemiconductorsEdgeAISoftware_patent_positive_bert.csv
Skipping AIMLAIMLSemiconductorsIntelligentSensorsDevices: output already exists at AIMLAIMLSemiconductorsIntelligentSensorsDevices_patent_positive_bert.csv
Skipping AIMLAIMLSemiconductorsProcessorDesign: output already exists at AIMLAIMLSemiconductorsProcessorDesign_patent_positive_bert.csv
Skipping AIMLAutonomousMachinesIntelligentRobotics: output already exists at AIMLAutonomousMachinesIntelligentRobotics_patent_positive_bert.csv
Skipping AIMLHorizontalPlatformsAIAutomationPlatforms: output already exists at AIMLHorizontalPlatformsAIAutomationPlatforms_patent_positive_bert.csv
Skipping AIMLHorizontalPlatformsAICore: output already exists at AIMLHorizontalPlatformsAICore_patent_positive_bert.csv
Skipping AIMLHorizontalPlatformsComputerVision: output already exists at AIMLHorizontalPlatformsComputerVision_patent_positive_bert.csv
Skipping AIMLHorizontalP

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


...

Processing subsegment: SupplyChainTechSupplyChainTechSupplyChainTech. Total number of sectors: 217


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Processing subsegment: SupplyChainTechWarehousingtechSustainablepackaging. Total number of sectors: 217


[('RetailHealthTechPersonalizedMedicineTestingBioinformatics',
        appid                                           abstract  pred_pos  \
  0   6864593  The invention is directed to a device for the ... -4.278506   
  1   8506297  A dental modal for making a dental prosthesis ... -3.727026   
  2  11297235  An optical apparatus configured to correct an ... -3.430620   
  3   9493045  A method is provided for reinforcement of a mu... -4.044676   
  4  10609247  An information processing system transmits and... -4.349476   
  
     pred_neg  positive                                             sector  
  0  3.157955         0  RetailHealthTechPersonalizedMedicineTestingBio...  
  1  2.675728         0  RetailHealthTechPersonalizedMedicineTestingBio...  
  2  2.378442         0  RetailHealthTechPersonalizedMedicineTestingBio...  
  3  3.017955         0  RetailHealthTechPersonalizedMedicineTestingBio...  
  4  3.190381         0  RetailHealthTechPersonalizedMedicineTestingBio...  ),
 (